# 01 CDRMIP Pipeline

In [1]:
%load_ext autoreload
%autoreload 2
    
from dask.distributed import Client
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

from cdrmip_extremes import (
    configs, load_pipeline, io_disk, gwl, utils,
    extremes_pipeline
)


In [2]:
client = Client(memory_limit=None,threads_per_worker=1,n_workers=28)

In [3]:
config = load_pipeline.LoadConfig(
    experiments=[
        load_pipeline.ExperimentConfig(
            experiment_id='1pctCO2-cdr', table_id='Amon', variable_id='tas',
            anomaly_reference='piControl',grid_label=None,
            overshoot=True, crossing_gwls = [1.5], gwl_match_period = (320,340)
        ),
    ],
    realign_years_overrides={'1pctCO2': (0, 140)}
)

In [4]:
pipe = load_pipeline.LoadingPipeline(config).run()

=== Loading 1pctCO2-cdr ===
Searching for files matching: {'experiment_id': '1pctCO2-cdr', 'table_id': 'Amon', 'variable_id': 'tas', 'grid_label': None, 'member_id': None}
Loading ACCESS-ESM1-5 / r1i1p1f1
  0241-01-16 12:00:00 -> 1000-12-16 12:00:00
Loading CESM2 / r1i1p1f1
  0001-01-15 12:00:00 -> 0200-12-15 12:00:00
Loading CNRM-ESM2-1 / r1i1p1f2
  1990-01-16 12:00:00 -> 2189-12-16 12:00:00
Loading CanESM5 / r1i1p2f1
  1991-01-16 12:00:00 -> 2290-12-16 12:00:00
Loading GFDL-ESM4 / r1i1p1f1
  0141-01-16 12:00:00 -> 0340-12-16 12:00:00
Loading MIROC-ES2L / r1i1p1f2
  1989-01-16 12:00:00 -> 2490-12-16 12:00:00
Loading NorESM2-LM / r1i1p1f1
  0141-01-16 12:00:00 -> 0399-12-16 12:00:00
Loading UKESM1-0-LL / r1i1p1f2
  1990-01-16 00:00:00 -> 2639-12-16 00:00:00
Searching for ACCESS-ESM1-5 1pctCO2-cdr r1i1p1f1 parent
Loading ACCESS-ESM1-5 1pctCO2 r1i1p1f1
Concatenating 1pctCO2-cdr -> 1pctCO2 [ACCESS-ESM1-5] via realign_years=(0, 140)
Concatenating via realign_years=(0, 140) (parent relabele

In [5]:
pipe.load_gsat().compute_gsat_rolling().compute_peak_warming()

In [6]:
pipe.compute_crossing_years().compute_gwl_matching()

## Save Anomalies

In [7]:
pipe.save_tree(
    'anomaly',
    'tas',
    proc_dir = configs.PROC_DIR,
    activity_id = 'CDRMIP',
    category = 'anomaly'
)

Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tas/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_tas_anomaly.nc
Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tas/CESM2/1pctCO2-cdr/CESM2_1pctCO2-cdr_r1i1p1f1_tas_anomaly.nc
Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tas/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_tas_anomaly.nc
Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tas/CanESM5/1pctCO2-cdr/CanESM5_1pctCO2-cdr_r1i1p2f1_tas_anomaly.nc
Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tas/GFDL-ESM4/1pctCO2-cdr/GFDL-ESM4_1pctCO2-cdr_r1i1p1f1_tas_anomaly.nc
Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tas/MIROC-ES2L/1pctCO2-cdr/MIROC-ES2L_1pctCO2-cdr_r1i1p1f2_tas_anomaly.nc
Skipping (already ex

In [29]:
pipe.save_tree(
    'gsat',
    'gsat',
    proc_dir = configs.PROC_DIR,
    activity_id = 'CDRMIP',
    category = None
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/gsat/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_gsat.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/gsat/CESM2/1pctCO2-cdr/CESM2_1pctCO2-cdr_r1i1p1f1_gsat.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/gsat/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_gsat.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/gsat/CanESM5/1pctCO2-cdr/CanESM5_1pctCO2-cdr_r1i1p2f1_gsat.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/gsat/GFDL-ESM4/1pctCO2-cdr/GFDL-ESM4_1pctCO2-cdr_r1i1p1f1_gsat.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/gsat/MIROC-ES2L/1pctCO2-cdr/MIROC-ES2L_1pctCO2-cdr_r1i1p1f2_gsat.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/gsat/NorESM2-LM/1pctCO2-cdr/NorESM2-LM_1pctCO2-cdr_r1i1p1f1_gsat.nc
Saving: /g/data/if69/sc1326/re

In [13]:
pipe.save_tree(
    'gsat_rolling',
    'gsat_rolling',
    proc_dir = configs.PROC_DIR,
    activity_id = 'CDRMIP',
    category = None,
    overwrite=True
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/gsat_rolling/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_gsat_rolling.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/gsat_rolling/CESM2/1pctCO2-cdr/CESM2_1pctCO2-cdr_r1i1p1f1_gsat_rolling.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/gsat_rolling/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_gsat_rolling.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/gsat_rolling/CanESM5/1pctCO2-cdr/CanESM5_1pctCO2-cdr_r1i1p2f1_gsat_rolling.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/gsat_rolling/GFDL-ESM4/1pctCO2-cdr/GFDL-ESM4_1pctCO2-cdr_r1i1p1f1_gsat_rolling.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/gsat_rolling/MIROC-ES2L/1pctCO2-cdr/MIROC-ES2L_1pctCO2-cdr_r1i1p1f2_gsat_rolling.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/gs

In [10]:
pipe.save_tree(
    'crossing_years',
    'crossing_years',
    proc_dir = configs.PROC_DIR,
    activity_id = 'CDRMIP',
    category = None,
    overwrite = True
)
pipe.save_tree(
    'gwl_matching',
    'gwl_matching_years',
    proc_dir = configs.PROC_DIR,
    activity_id = 'CDRMIP',
    category = None,
    overwrite = True
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/crossing_years/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_crossing_years.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/crossing_years/CESM2/1pctCO2-cdr/CESM2_1pctCO2-cdr_r1i1p1f1_crossing_years.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/crossing_years/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_crossing_years.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/crossing_years/CanESM5/1pctCO2-cdr/CanESM5_1pctCO2-cdr_r1i1p2f1_crossing_years.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/crossing_years/GFDL-ESM4/1pctCO2-cdr/GFDL-ESM4_1pctCO2-cdr_r1i1p1f1_crossing_years.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/crossing_years/MIROC-ES2L/1pctCO2-cdr/MIROC-ES2L_1pctCO2-cdr_r1i1p1f2_crossing_years.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/

In [8]:
io_disk.save_peak_warming(
    pipe.peak_warming,
    'CDRMIP',
    configs.PROC_DIR
)

## GWL Analysis

In [8]:
diffs = utils.map_nested(
    lambda ds, gy: gwl.gwl_branch_difference(gwl.extract_gwl_period(ds, gy, window=21)),
    pipe.anomaly, pipe.crossing_years,
)

In [9]:
matching_diffs = utils.map_nested(
    lambda ds, gy: gwl.gwl_branch_difference(gwl.extract_gwl_period(ds, gy, window=21)),
    pipe.anomaly, pipe.gwl_matching,
)

In [24]:
io_disk.save_tree(
    diffs,
    configs.PROC_DIR,
    'tas_gwl_differences',
    activity_id='CDRMIP',
    category=None
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tas_gwl_differences/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_tas_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tas_gwl_differences/CESM2/1pctCO2-cdr/CESM2_1pctCO2-cdr_r1i1p1f1_tas_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tas_gwl_differences/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_tas_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tas_gwl_differences/CanESM5/1pctCO2-cdr/CanESM5_1pctCO2-cdr_r1i1p2f1_tas_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tas_gwl_differences/GFDL-ESM4/1pctCO2-cdr/GFDL-ESM4_1pctCO2-cdr_r1i1p1f1_tas_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tas_gwl_differences/MIROC-ES2L/1pctCO2-cdr/MIROC-ES2L_1pctCO2-cdr_r1i1p1f2_tas_gwl_diffe

In [25]:
io_disk.save_tree(
    matching_diffs,
    configs.PROC_DIR,
    'tas_matching_gwl_differences',
    activity_id='CDRMIP',
    category=None
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tas_matching_gwl_differences/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_tas_matching_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tas_matching_gwl_differences/CESM2/1pctCO2-cdr/CESM2_1pctCO2-cdr_r1i1p1f1_tas_matching_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tas_matching_gwl_differences/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_tas_matching_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tas_matching_gwl_differences/CanESM5/1pctCO2-cdr/CanESM5_1pctCO2-cdr_r1i1p2f1_tas_matching_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/tas_matching_gwl_differences/GFDL-ESM4/1pctCO2-cdr/GFDL-ESM4_1pctCO2-cdr_r1i1p1f1_tas_matching_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMI

## Extreme Temperature Frequency Analysis

In [7]:
ext = extremes_pipeline.ExtremesPipeline(
    regridded_piControl=pipe.regridded_piControl,
    regridded_concat=pipe.regridded_concat,
    crossing_years=pipe.crossing_years,
    gwl_matching=pipe.gwl_matching,
    gwl_window=21
)    

In [8]:
ext.run()

In [9]:
ext.load(attribute='exceedances')
ext.load(attribute='exceedances_matching')
ext.compute_multi_model_stat('exceedances')
ext.compute_multi_model_stat('exceedances_matching')

In [10]:
ext.compute_branch_differences(source_attr='exceedances_matching')
ext.compute_multi_model_stat('exceedances_matching')
ext.compute_multi_model_stat('branch_differences')
ext.compute_multi_model_stat('branch_differences_matching')

In [13]:
ext.compute_agreement(source_attr='branch_differences_matching')

In [15]:
ext.save(
    configs.PROC_DIR,
    'CDRMIP',
    outputs=[
        'agreement',
        'exceedances_median_multimodel',
        'branch_differences_median_multimodel',
        'gwl_period_extremes',
        'agreement_matching',
        'exceedances_matching_median_multimodel',
        'branch_differences_matching_median_multimodel',
        'gwl_period_extremes_matching',
        'thresholds',
        'agreement_matching',
        'extreme_months'
        
    ],
    overwrite=False
)

Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/extremes/agreement/1pctCO2-cdr/1pctCO2-cdr_heat_tas_agreement.nc
Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/extremes/agreement/1pctCO2-cdr/1pctCO2-cdr_cold_tas_agreement.nc
Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/extremes/exceedances_median_multimodel/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_heat_tas_exceedances_median_multimodel.nc
Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/extremes/exceedances_median_multimodel/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_cold_tas_exceedances_median_multimodel.nc
Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/extremes/exceedances_median_multimodel/CESM2/1pctCO2-cdr/CESM2_1pctCO2-cdr_heat_tas_exceedances_median_multimodel.nc
Skipping (already exists): /g/da

## Repeat Extremes Analysis at SSP GWLs

In [15]:
ssp_data =  io_disk.load_mip('ScenarioMIP', None, skip={'crossing_years'})

In [19]:
# overlap models
models = [
    'ACCESS-ESM1-5',
    'CanESM5',
    'CNRM-ESM2-1',
    'MIROC-ES2L',
    'UKESM1-0-LL',
]

In [21]:
ssp_crossing_years = defaultdict(lambda: defaultdict(dict))
for model in models:
    member_dt = pipe.gsat[model]['1pctCO2-cdr']
    gwls = ssp_data['gwl_matching_years'][model]['ssp534-over'].gwl.values
    print(model)
    print(gwls)
    
    for member, gsat_da in member_dt.items():
        ssp_crossing_years[model]['1pctCO2-cdr'][member] = gwl.find_crossing_years(gsat_da.gsat,window=21,gwls=gwls)
ssp_crossing_years = utils.defaultdict_to_dict(ssp_crossing_years)
ssp_crossing_years

ACCESS-ESM1-5
[1.84]
CanESM5
[2.39]
CNRM-ESM2-1
[2.04]
MIROC-ES2L
[1.23]
UKESM1-0-LL
[2.44]


{'ACCESS-ESM1-5': {'1pctCO2-cdr': {'r1i1p1f1': <xarray.DataArray (gwl: 1, branch: 2)> Size: 16B
   array([[ 68, 254]])
   Coordinates:
     * gwl      (gwl) float64 8B 1.84
     * branch   (branch) <U9 72B 'ramp_up' 'ramp_down'}},
 'CanESM5': {'1pctCO2-cdr': {'r1i1p2f1': <xarray.DataArray (gwl: 1, branch: 2)> Size: 16B
   array([[ 65, 261]])
   Coordinates:
     * gwl      (gwl) float64 8B 2.39
     * branch   (branch) <U9 72B 'ramp_up' 'ramp_down'}},
 'CNRM-ESM2-1': {'1pctCO2-cdr': {'r1i1p1f2': <xarray.DataArray (gwl: 1, branch: 2)> Size: 16B
   array([[ 73, 255]])
   Coordinates:
     * gwl      (gwl) float64 8B 2.04
     * branch   (branch) <U9 72B 'ramp_up' 'ramp_down'}},
 'MIROC-ES2L': {'1pctCO2-cdr': {'r1i1p1f2': <xarray.DataArray (gwl: 1, branch: 2)> Size: 16B
   array([[ 59, 246]])
   Coordinates:
     * gwl      (gwl) float64 8B 1.23
     * branch   (branch) <U9 72B 'ramp_up' 'ramp_down'}},
 'UKESM1-0-LL': {'1pctCO2-cdr': {'r1i1p1f2': <xarray.DataArray (gwl: 1, branch: 2)> Siz

In [30]:
ext_ssp_compare = extremes_pipeline.ExtremesPipeline(
    regridded_piControl={
        model: member_dt for model, member_dt in pipe.regridded_piControl.items() if model in models
    },
    regridded_concat={
        model: expt_dt for model, expt_dt in pipe.regridded_concat.items() if model in models
    },
    crossing_years=ssp_crossing_years,
    gwl_window=21
) 

In [31]:
ext_ssp_compare.run()

  No gwl_matching provided — skipping gwl_period_extremes_matching


In [32]:
ext_ssp_compare.load(attribute='exceedances')
ext_ssp_compare.compute_multi_model_stat('exceedances')
ext_ssp_compare.compute_multi_model_stat('branch_differences')

In [34]:
ext_ssp_compare.save(
    configs.PROC_DIR,
    'CDRMIP',
    outputs=[
        'agreement',
        'exceedances_median_multimodel',
        'branch_differences_median_multimodel',
        'gwl_period_extremes',
    ],
    overwrite=False,
    category='extremes_compare'
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/extremes_compare/agreement/1pctCO2-cdr/1pctCO2-cdr_heat_tas_agreement.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/extremes_compare/agreement/1pctCO2-cdr/1pctCO2-cdr_cold_tas_agreement.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/extremes_compare/exceedances_median_multimodel/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_heat_tas_exceedances_median_multimodel.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/extremes_compare/exceedances_median_multimodel/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_cold_tas_exceedances_median_multimodel.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/extremes_compare/exceedances_median_multimodel/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_heat_tas_exceedances_median_multimodel.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/ext